In [1]:
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Step 37 — Car skims for 2026: the V2 areas routed over the May 2026 car link speeds (`Car_skim_2026_network.ipynb`, task C1 / E5)

The car in-vehicle time that the generalized-cost inventory (step 26, §6x), the mode skims
(step 31, §6ac) and the capture (step 32, §6ad) use is the **2017/18 survey's door-to-door
time**, allocated to the 25 V2 areas and smoothed. Task C1 / E5 asked for a 2026 car time per
area pair, to measure the uplift on that skim and to run the capture on the uplifted skim; the
route first specified (the Google Distance Matrix API) proved structurally unavailable for a
month already in the past (`docs/NEXT_STEPS_HANDOVER_2026-09-23.md` C1), and the item waited
for a car-network source. That source arrived on 5 October 2026 and was prepared by step 43
(§6al): the May 2026 car speed network clipped to the study area, a speed per link, hour and
direction (`Output/car_speed/`).

This step **routes every V2 area pair at the 07:00–08:00 car speeds** and compares the result
with the survey skim pair by pair. It writes the network car skim beside the survey one, and
step 26 gains a switch `CAR_SOURCE = 'network'` that uses it (plus a stated terminal
allowance) in place of the survey time, so that 26 → 31 → 32 can be rerun on the 2026 car
times without touching the default outputs.

**Why the routing runs on the bus street network.** The car layer is the bus-speed street
network with its segments merged into longer links (step 43 §4), and the merge did not keep
the junctions: at a metre's tolerance only 22 % of its end points form a connected network,
the typical gap at a junction being 10–20 m (the two carriageways of a divided road are
separate one-way links that the cross streets do not reach). Rather than snap end points by a
tolerance that would also join grade-separated and parallel roads, the step keeps the
**street network's own topology** — the one step 26 routed the bus on, 54,507 segments in the
study box — and **transfers each car link's speeds onto the street segments that lie on it**
(both segment end points within 3 m of the car link; the segment's drawing direction against
the link's decides which of `AB` / `BA` applies). 85 % of the street length in the box lies on a
car link and 68 % carries an observed 07:00 speed; the rest is local streets. As a
cross-check the car layer is also routed on its own, with its end points snapped at 50 m.

**Stated assumptions.** Street segments without a transferred speed carry an **assumed
30 km/h** (the median observed link speed is 31.5 km/h and these are the local streets); the
share of each path on such segments is reported, and the skim is reproduced at 20 and
40 km/h. The area's representative point is the one step 26 uses for its centroid distances
(`Output/lrt_v2/lrt_area_representative_station.csv`), attached to the nearest node of the
giant strongly-connected component by a straight-line connector at 20 km/h. The network time
is kerb to kerb; step 26 adds a **3-minute terminal allowance** (parking and walking at both
ends, the hand-over's figure) when it substitutes the survey's door-to-door time. The 07:00
column is taken as the peak speed with caveat 24 in force: the delivery's hourly profile is
almost flat, so the "peak" time is in practice a May 2026 all-day time (the fastest-hour
skim is reported beside it).

In [2]:
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import shapefile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import dijkstra, connected_components
from scipy.spatial import cKDTree

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/gc'; FIG = 'Output/figures'; os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)
CAR = 'Output/car_speed/GoogleSpeed_202605_North.gpkg'          # step 43: the May 2026 car speed network clipped to TAZ_North
STREETS = 'Input/BusSpeedData/Streets/Streets'                   # the bus-speed street network (geometry and DIR only; no bus speeds used here)
HOUR = 7                                                         # 07:00–08:00: SPD_7_AB / SPD_7_BA
FALLBACK_KMH = 30.0                                              # assumed speed on segments without a transferred car speed (stated; sensitivity below)
CONNECTOR_KMH = 20.0                                             # area point -> nearest network node, straight line
TERMINAL_MIN = 3.0                                               # parking + walk at both ends, added by step 26 (not here) — the hand-over's allowance
MATCH_TOL = 3.0                                                  # m: a street segment lies on a car link when both its end points are within this distance
TRUNK = list(range(201, 211))                                    # the 90 trunk pairs of §6x

xl = pd.ExcelFile('Input/Corridor_TAZ_Agg_V2.xlsx'); areas = xl.parse('AreaCodes').set_index('AggCode')
AREAS = list(areas.index); names = areas['AggAreaName']; A = len(AREAS)
ac = pd.read_csv('Output/lrt_v2/lrt_area_representative_station.csv', index_col=0).reindex(AREAS)
DIST_KM = pd.DataFrame(np.hypot(ac['cx'].values[:, None] - ac['cx'].values[None, :], ac['cy'].values[:, None] - ac['cy'].values[None, :]) / 1000, index=AREAS, columns=AREAS)
car = gpd.read_file(CAR, engine='pyogrio')
HOURS = sorted({int(c.split('_')[1]) for c in car.columns if c.startswith('SPD_')})
for d in ('AB', 'BA'):
    car[f'v7_{d}'] = car[f'SPD_{HOUR}_{d}'].fillna(0); car[f'vmax_{d}'] = car[[f'SPD_{h}_{d}' for h in HOURS]].fillna(0).max(axis=1)   # fastest hour: a free-flow proxy
tz = shapefile.Reader('Input/TAZ_North/TAZ_North'); BB = tz.bbox
t0 = time.time(); rd = shapefile.Reader(STREETS); segs = []
for s, rec in zip(rd.iterShapes(), rd.iterRecords()):
    b = s.bbox
    if not (b[2] > BB[0] and b[0] < BB[2] and b[3] > BB[1] and b[1] < BB[3]): continue
    segs.append((LineString(s.points), rec['DIR'], rec['USERID']))
print(f'{len(car):,} car links ({car.geometry.length.sum() / 1000:,.0f} km); {len(segs):,} street segments in the study box ({sum(s[0].length for s in segs) / 1000:,.0f} km); {A} areas, centroid distances up to {DIST_KM.values.max():.1f} km; {time.time() - t0:.0f} s')

9,927 car links (4,996 km); 54,507 street segments in the study box (5,632 km); 25 areas, centroid distances up to 35.0 km; 5 s


## 1. The car speeds transferred onto the street segments

For every street segment the car links within 3 m are tested; the first whose geometry holds
both end points of the segment (and is at least as long) is its car link. The segment's
vertex order against the car link's decides which car direction is its `AB`. The result is a
07:00 speed and a fastest-hour speed per segment and direction, zero where the car link has
no observation in that direction or the segment has no car link.

In [3]:
t0 = time.time(); tree = STRtree(car.geometry.values); G_car = car.geometry.values; L_car = car.geometry.length.values
match = np.full(len(segs), -1); same = np.ones(len(segs), bool)
for k, (L, dr, uid) in enumerate(segs):
    a, b = Point(L.coords[0]), Point(L.coords[-1])
    for c in tree.query(L.buffer(MATCH_TOL)):
        g = G_car[c]
        if g.distance(a) < MATCH_TOL and g.distance(b) < MATCH_TOL and L.length <= L_car[c] + MATCH_TOL:
            match[k] = c; same[k] = g.project(a) < g.project(b); break
e = pd.DataFrame({'ax': [s[0].coords[0][0] for s in segs], 'ay': [s[0].coords[0][1] for s in segs], 'bx': [s[0].coords[-1][0] for s in segs], 'by': [s[0].coords[-1][1] for s in segs],
                  'len': [s[0].length for s in segs], 'dir': [s[1] for s in segs], 'uid': [s[2] for s in segs], 'car': match, 'same': same})
m = match >= 0; idx = np.where(m, match, 0)
for v in ('v7', 'vmax'):
    ab, ba = car[f'{v}_AB'].values[idx], car[f'{v}_BA'].values[idx]
    e[f'{v}_AB'] = np.where(m, np.where(same, ab, ba), 0.0); e[f'{v}_BA'] = np.where(m, np.where(same, ba, ab), 0.0)
obs = (e['v7_AB'] > 0) | (e['v7_BA'] > 0)
car_dir_eff = np.where(same, car['DIR'].values[idx], -car['DIR'].values[idx])
print(f"segments on a car link: {m.sum():,} of {len(e):,} ({e.loc[m, 'len'].sum() / e['len'].sum():.0%} of the length); with an observed 07:00 car speed: {obs.sum():,} ({e.loc[obs, 'len'].sum() / e['len'].sum():.0%} of the length); "
      f"car links reached by at least one segment: {len(set(match[m])):,} of {len(car):,}; DIR agreement segment vs car link (orientation allowed for): {(car_dir_eff[m] == e.loc[m, 'dir'].values).mean():.1%}; {time.time() - t0:.0f} s")
cov = pd.DataFrame({'segments': [len(e), int(m.sum()), int(obs.sum())], 'km': [e['len'].sum() / 1000, e.loc[m, 'len'].sum() / 1000, e.loc[obs, 'len'].sum() / 1000]}, index=['street segments in the study box', 'on a car link', 'with an observed 07:00 car speed'])
cov.to_csv(f'{OUT}/car_network_segment_coverage.csv', float_format='%.1f')

segments on a car link: 38,060 of 54,507 (85% of the length); with an observed 07:00 car speed: 25,029 (68% of the length); car links reached by at least one segment: 9,560 of 9,927; DIR agreement segment vs car link (orientation allowed for): 96.6%; 11 s


## 2. The directed graph and the routing

Nodes are the segment end points rounded to a metre (step 26's rule); a segment contributes an
edge in each direction its `DIR` allows, timed at its transferred 07:00 car speed in that
direction, or at the fallback speed when open but without a speed. The area points attach to
their nearest node of the giant strongly-connected component; one Dijkstra per area gives the
25 × 25 times, and the predecessor chains the path length and the length on fallback segments.

In [4]:
nodes = {}
def node(x, y):
    k = (round(x), round(y))
    if k not in nodes: nodes[k] = len(nodes)
    return nodes[k]
e['a'] = [node(x, y) for x, y in zip(e['ax'], e['ay'])]; e['b'] = [node(x, y) for x, y in zip(e['bx'], e['by'])]
N = len(nodes); XY = np.array([k for k, v in sorted(nodes.items(), key=lambda kv: kv[1])], dtype=float)

def build(E, vcol='v7', fallback=FALLBACK_KMH):
    rows, cols, tmin, lens, fb = [], [], [], [], []
    for a, b, ln, dr, vab, vba in zip(E['a'], E['b'], E['len'], E['dir'], E[f'{vcol}_AB'], E[f'{vcol}_BA']):
        for (u, w, v, open_) in ((a, b, vab, dr in (0, 1)), (b, a, vba, dr in (0, -1))):
            if not open_: continue
            if v > 0: speed, f = v, 0.0
            elif fallback is None: continue
            else: speed, f = fallback, ln
            rows.append(u); cols.append(w); tmin.append(ln / 1000 / speed * 60); lens.append(ln); fb.append(f)
    n = max(E['a'].max(), E['b'].max()) + 1
    G = coo_matrix((tmin, (rows, cols)), shape=(n, n)).tocsr(); GL = coo_matrix((lens, (rows, cols)), shape=(n, n)).tocsr(); GF = coo_matrix((fb, (rows, cols)), shape=(n, n)).tocsr()
    ncomp, lab = connected_components(G, directed=True, connection='strong'); giant = np.bincount(lab).argmax()
    used = set(rows) | set(cols); served = np.array(sorted(v for v in used if lab[v] == giant))
    return G, GL, GF, served, len(used)

def skim(E=e, xy=None, vcol='v7', fallback=FALLBACK_KMH, connector_kmh=CONNECTOR_KMH):
    xy = XY if xy is None else xy
    G, GL, GF, served, nused = build(E, vcol, fallback)
    kdt = cKDTree(xy[served]); conn_d, conn_i = kdt.query(ac[['cx', 'cy']].values); anchor = served[conn_i]
    Dt, pred = dijkstra(G, indices=anchor, directed=True, return_predecessors=True)
    T = Dt[:, anchor] + (conn_d[:, None] + conn_d[None, :]) / 1000 / connector_kmh * 60 * (1 - np.eye(A))
    def walk(i, j, M):
        if i == j: return 0.0
        k, tot = anchor[j], 0.0
        while k != anchor[i]:
            p = pred[i, k]
            if p < 0: return np.nan
            tot += M[p, k]; k = p
        return tot
    L = np.array([[walk(i, j, GL) for j in range(A)] for i in range(A)]) / 1000
    F = np.array([[walk(i, j, GF) for j in range(A)] for i in range(A)]) / 1000
    info = {'nodes_used': nused, 'giant': len(served), 'conn_min': conn_d.min(), 'conn_max': conn_d.max(), 'conn_mean': conn_d.mean()}
    return pd.DataFrame(T, index=AREAS, columns=AREAS), pd.DataFrame(L, index=AREAS, columns=AREAS), pd.DataFrame(F, index=AREAS, columns=AREAS), info, pd.Series(conn_d, index=AREAS)

t0 = time.time(); car_t, car_km, car_fb, info, conn = skim()
print(f"{N:,} nodes, {info['nodes_used']:,} on an open edge, giant strongly-connected component {info['giant']:,} ({info['giant'] / info['nodes_used']:.0%}); connectors {info['conn_min']:.0f}–{info['conn_max']:.0f} m (mean {info['conn_mean']:.0f}); {time.time() - t0:.0f} s")
assert np.isfinite(car_t.values).all(), 'an area pair is unreachable'
off = ~np.eye(A, dtype=bool); fb_share = (car_fb / car_km.replace(0, np.nan)).fillna(0)
print(f'07:00 network time, off-diagonal pairs: {car_t.values[off].min():.1f}–{car_t.values[off].max():.1f} min; path length {car_km.values[off].min():.1f}–{car_km.values[off].max():.1f} km; '
      f'share of the path on fallback segments: median {np.median(fb_share.values[off]):.1%}, p90 {np.quantile(fb_share.values[off], .9):.1%}, max {fb_share.values[off].max():.1%}')
tk = ~np.eye(len(TRUNK), dtype=bool)
print(f'implied kerb-to-kerb speed on the trunk pairs: {60 * np.mean(car_km.loc[TRUNK, TRUNK].values[tk] / car_t.loc[TRUNK, TRUNK].values[tk]):.1f} km/h (path km / network min, simple mean)')
car_t.to_csv(f'{OUT}/car_ivt_network_area_v2.csv', float_format='%.2f'); car_km.to_csv(f'{OUT}/car_km_network_area_v2.csv', float_format='%.2f'); fb_share.to_csv(f'{OUT}/car_path_fallback_share_area_v2.csv', float_format='%.3f')
conn.rename('connector_m').to_frame().assign(name=names).to_csv(f'{OUT}/car_network_area_connectors.csv', float_format='%.0f')
car_t_ff = skim(vcol='vmax')[0]; car_t_ff.to_csv(f'{OUT}/car_ivt_network_fastest_hour_area_v2.csv', float_format='%.2f')
ratio_ff = (car_t / car_t_ff).values[off]
print(f'07:00 ÷ fastest-hour time: median {np.median(ratio_ff):.3f}, max {ratio_ff.max():.3f} (the free-flow proxy; caveat 24: the hourly profile is almost flat)')

48,898 nodes, 48,898 on an open edge, giant strongly-connected component 47,609 (97%); connectors 16–625 m (mean 96); 2 s
07:00 network time, off-diagonal pairs: 2.9–61.6 min; path length 1.3–54.1 km; share of the path on fallback segments: median 23.7%, p90 45.1%, max 93.7%
implied kerb-to-kerb speed on the trunk pairs: 37.9 km/h (path km / network min, simple mean)


07:00 ÷ fastest-hour time: median 1.039, max 1.186 (the free-flow proxy; caveat 24: the hourly profile is almost flat)


## 3. Checks: symmetry, the fallback assumption, the car layer routed on its own

Every pair is reachable; the two directions of a pair are compared; the trunk-pair time is
reproduced with the fallback speed at 20 and 40 km/h and at the fastest hour; and, as the
cross-check of the transfer, the car layer itself is routed with its end points snapped at
50 m (99 % of its nodes then connect) — a tolerance that also joins some roads that do not
meet, so it is a check, not the method.

In [5]:
with np.errstate(divide='ignore', invalid='ignore'): sym = (car_t.values / car_t.values.T)[off]
print(f'directional asymmetry (t_od / t_do): median {np.median(sym):.3f}, 5–95 % {np.quantile(sym, .05):.2f}–{np.quantile(sym, .95):.2f}; pairs differing by > 30 %: {int((np.abs(sym - 1) > .3).sum())} of {off.sum()}')
assert (car_t_ff.values <= car_t.values + 1e-9).all()
flows = pd.read_csv('Output/skims/pair_flows_and_skims_2022.csv')
W = flows.pivot(index='o', columns='d', values='car_2022').reindex(index=AREAS, columns=AREAS).fillna(0)
def trunk_mean(M): return np.average(M.loc[TRUNK, TRUNK].values, weights=W.loc[TRUNK, TRUNK].values * tk)
def all_mean(M): return np.average(M.values, weights=W.values * off)
# the car layer on its own topology, end points snapped by a 50 m union-find
P = np.vstack([np.array([g.coords[0] for g in car.geometry]), np.array([g.coords[-1] for g in car.geometry])]); nc_ = len(car)
parent = np.arange(len(P))
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
for a_, b_ in cKDTree(P).query_pairs(50.0, output_type='ndarray'):
    ra, rb = find(a_), find(b_)
    if ra != rb: parent[ra] = rb
lab = np.array([find(x) for x in range(len(P))]); _, lab = np.unique(lab, return_inverse=True)
e_car = pd.DataFrame({'a': lab[:nc_], 'b': lab[nc_:], 'len': car.geometry.length.values, 'dir': car['DIR'].values, 'v7_AB': car['v7_AB'].values, 'v7_BA': car['v7_BA'].values})
xy_car = np.zeros((lab.max() + 1, 2)); np.add.at(xy_car, lab, P); xy_car /= np.bincount(lab)[:, None]
car_t_own, _, _, info_own, _ = skim(E=e_car, xy=xy_car)
print(f"car layer routed on its own (50 m snap): {lab.max() + 1:,} nodes, giant component {info_own['giant']:,} ({info_own['giant'] / info_own['nodes_used']:.0%}), connectors up to {info_own['conn_max']:.0f} m")
variants = [('07:00, fallback 30 km/h (central)', car_t), ('07:00, fallback 20 km/h', skim(fallback=20.0)[0]), ('07:00, fallback 40 km/h', skim(fallback=40.0)[0]),
            ('fastest hour, fallback 30 km/h', car_t_ff), ('cross-check: car layer on its own topology, 50 m snap, fallback 30', car_t_own)]
sens = pd.DataFrame([{'variant': k, 'trunk pairs, trip-weighted (min)': trunk_mean(M), 'all pairs, trip-weighted (min)': all_mean(M), 'trunk pairs, simple mean (min)': M.loc[TRUNK, TRUNK].values[tk].mean()} for k, M in variants])
sens.to_csv(f'{OUT}/car_network_skim_sensitivity.csv', index=False, float_format='%.2f'); print(sens.round(2).to_string(index=False))

directional asymmetry (t_od / t_do): median 1.000, 5–95 % 0.68–1.48; pairs differing by > 30 %: 96 of 600


car layer routed on its own (50 m snap): 5,059 nodes, giant component 4,938 (98%), connectors up to 625 m


                                                           variant  trunk pairs, trip-weighted (min)  all pairs, trip-weighted (min)  trunk pairs, simple mean (min)
                                 07:00, fallback 30 km/h (central)                              9.96                           13.08                           11.91
                                           07:00, fallback 20 km/h                             11.48                           15.20                           13.63
                                           07:00, fallback 40 km/h                              9.08                           11.77                           10.88
                                    fastest hour, fallback 30 km/h                              9.46                           12.52                           11.37
cross-check: car layer on its own topology, 50 m snap, fallback 30                              8.80                           12.19                           11.15


## 4. Against the survey skim

The survey car skim of step 26 (`car_ivt_survey_area_v2.csv`: door-to-door `TrvlTime` of the
2017/18 AM car trips, EB-smoothed with the distance fit where fewer than 10 are sampled) is
compared pair by pair with the network time **plus the 3-minute terminal allowance** — the
quantity step 26 substitutes under `CAR_SOURCE = 'network'`. The ratio is reported by
centroid-distance band, on the 90 trunk pairs trip-weighted by the 2022 car flows, and on the
pairs where the survey has at least 10 sampled trips (its own "measured" cells). The survey time
includes parking and walking and is a reported, rounded duration; the network time is a
fastest path at measured speeds between fixed representative points, with no intersection
delay beyond what the speeds carry. The two are different quantities, and the band table is
where that shows.

In [6]:
surv = pd.read_csv('Output/gc/car_ivt_survey_area_v2.csv', index_col=0); surv.columns = surv.columns.astype(int); surv = surv.reindex(index=AREAS, columns=AREAS)
surv_n = pd.read_csv('Output/gc/car_ivt_n_sampled_area_v2.csv', index_col=0); surv_n.columns = surv_n.columns.astype(int); surv_n = surv_n.reindex(index=AREAS, columns=AREAS).fillna(0)
surv_km = pd.read_csv('Output/gc/car_dist_survey_area_v2.csv', index_col=0); surv_km.columns = surv_km.columns.astype(int); surv_km = surv_km.reindex(index=AREAS, columns=AREAS)
car_2026 = car_t + TERMINAL_MIN * off
rows = []
for o in AREAS:
    for d_ in AREAS:
        if o == d_: continue
        rows.append({'o': o, 'd': d_, 'o_name': names[o], 'd_name': names[d_], 'dist_km': DIST_KM.loc[o, d_], 'car_2022_trips': W.loc[o, d_], 'trunk': o in TRUNK and d_ in TRUNK,
                     'survey_min': surv.loc[o, d_], 'survey_n_sampled': surv_n.loc[o, d_], 'survey_km': surv_km.loc[o, d_],
                     'network_min_0700': car_t.loc[o, d_], 'network_km': car_km.loc[o, d_], 'fallback_share': fb_share.loc[o, d_], 'network_plus_terminal_min': car_2026.loc[o, d_], 'fastest_hour_min': car_t_ff.loc[o, d_]})
pairs = pd.DataFrame(rows); pairs['ratio_2026_over_survey'] = pairs['network_plus_terminal_min'] / pairs['survey_min']
pairs.to_csv(f'{OUT}/car_network_vs_survey_pairs.csv', index=False, float_format='%.3f')
BANDS = [(0, 3), (3, 6), (6, 10), (10, 20), (20, 99)]
def summ(x, label):
    w = x['car_2022_trips'].values; ok = w.sum() > 0
    return {'pairs': label, 'n_pairs': len(x), 'survey_min (trip-weighted)': np.average(x['survey_min'], weights=w) if ok else np.nan, 'network+3 min (trip-weighted)': np.average(x['network_plus_terminal_min'], weights=w) if ok else np.nan,
            'ratio (weighted means)': (np.average(x['network_plus_terminal_min'], weights=w) / np.average(x['survey_min'], weights=w)) if ok else np.nan,
            'median pair ratio': x['ratio_2026_over_survey'].median(), 'survey_min (simple mean)': x['survey_min'].mean(), 'network+3 (simple mean)': x['network_plus_terminal_min'].mean(),
            'network km (simple mean)': x['network_km'].mean(), 'survey km (simple mean)': x['survey_km'].mean(), 'car trips 2022': w.sum()}
summary = pd.DataFrame([summ(pairs[(pairs['dist_km'] >= lo) & (pairs['dist_km'] < hi)], f'{lo}–{hi} km' if hi < 99 else f'> {lo} km') for lo, hi in BANDS]
                       + [summ(pairs[pairs['trunk']], 'trunk pairs (201–210)'), summ(pairs[pairs['survey_n_sampled'] >= 10], 'survey measured cells (≥ 10 sampled)'), summ(pairs, 'all 600 pairs')])
summary.to_csv(f'{OUT}/car_network_vs_survey_summary.csv', index=False, float_format='%.3f'); print(summary.round(2).to_string(index=False))
tr = pairs[pairs['trunk']]; wt = tr['car_2022_trips'].values
print(f"trunk pairs, trip-weighted: survey {np.average(tr['survey_min'], weights=wt):.1f} min door-to-door vs network {np.average(tr['network_min_0700'], weights=wt):.1f} + {TERMINAL_MIN:.0f} = {np.average(tr['network_plus_terminal_min'], weights=wt):.1f} min "
      f"(ratio {np.average(tr['network_plus_terminal_min'], weights=wt) / np.average(tr['survey_min'], weights=wt):.2f}); reported survey km {tr['survey_km'].mean():.1f} vs path km {tr['network_km'].mean():.1f} (simple means)")

                               pairs  n_pairs  survey_min (trip-weighted)  network+3 min (trip-weighted)  ratio (weighted means)  median pair ratio  survey_min (simple mean)  network+3 (simple mean)  network km (simple mean)  survey km (simple mean)  car trips 2022
                              0–3 km       84                        9.26                           9.29                    1.00               0.98                     10.20                    10.24                      4.01                     1.73        26583.75
                              3–6 km      112                       14.28                          15.00                    1.05               0.99                     15.32                    15.84                      7.93                     3.95        11818.34
                             6–10 km      162                       23.51                          22.09                    0.94               0.95                     22.34                    22.17    

In [7]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 5))
m10 = pairs['survey_n_sampled'] >= 10
ax[0].scatter(pairs.loc[~m10, 'survey_min'], pairs.loc[~m10, 'network_plus_terminal_min'], s=10, color=MUTED, alpha=.5, label='survey cell smoothed (< 10 sampled)')
ax[0].scatter(pairs.loc[m10, 'survey_min'], pairs.loc[m10, 'network_plus_terminal_min'], s=14 + pairs.loc[m10, 'car_2022_trips'] / 60, color=BLUE, alpha=.7, label='survey cell measured (≥ 10 sampled; size = 2022 car trips)')
lim = max(pairs['survey_min'].max(), pairs['network_plus_terminal_min'].max()) * 1.05
ax[0].plot([0, lim], [0, lim], color=INK2, lw=.8, ls='--'); ax[0].set_xlim(0, lim); ax[0].set_ylim(0, lim)
ax[0].set_xlabel('survey 2017/18 door-to-door (min)'); ax[0].set_ylabel('May 2026 network 07:00 + 3 min terminal (min)'); ax[0].set_title('Car time per V2 area pair: network vs survey'); ax[0].legend(frameon=False, fontsize=8)
b = summary.iloc[:len(BANDS)]
ax[1].bar(range(len(b)), b['ratio (weighted means)'], color=AQUA, width=.6); ax[1].axhline(1, color=INK2, lw=.8, ls='--')
ax[1].set_xticks(range(len(b))); ax[1].set_xticklabels(b['pairs']); ax[1].set_ylabel('network + 3 min  ÷  survey (trip-weighted means)'); ax[1].set_title('Ratio by centroid-distance band')
for i, (r, n) in enumerate(zip(b['ratio (weighted means)'], b['n_pairs'])): ax[1].text(i, r + .02, f'{r:.2f}\n({n} pairs)', ha='center', fontsize=8)
for a in ax:
    for sp in ('top', 'right'): a.spines[sp].set_visible(False)
    a.grid(color=GRID, lw=.6)
plt.tight_layout(); plt.savefig(f'{FIG}/car_network_vs_survey.png', dpi=150); plt.close(); print('figure written')

figure written


## 5. Results, outputs

**The routing.** 48,898 nodes, 97 % in the giant strongly-connected component; the area
points attach within 16–625 m (mean 96 m). Off-diagonal times run 2.9–61.6 min over paths of
1.3–54.1 km; the implied kerb-to-kerb speed on the trunk pairs is 37.9 km/h. A median 24 % of
a path's length (p90 45 %) runs on segments without a transferred car speed — the local
streets at the two ends — which is why the fallback speed matters: the trunk trip-weighted
time is 10.0 min at 30 km/h, 11.5 at 20 and 9.1 at 40. The fastest-hour skim is 4 % faster
than the 07:00 one (the delivery's flat profile, caveat 24). The car layer routed on its own
snapped topology gives 8.8 min on the trunk against 10.0 here — the same order, the
difference being the snapped junctions and the local streets the car layer does not carry.

**Against the survey.** On the 90 trunk pairs, trip-weighted by the 2022 car flows, the survey
door-to-door time is **14.4 min** and the network time **10.0 + 3 = 13.0 min**, a ratio of
**0.90**; on the 20 cells the survey measures with ≥ 10 sampled trips 11.3 vs 10.9 (0.97); on
all 600 pairs 16.9 vs 16.1 (0.95). By centroid distance the ratio runs 1.00 (< 3 km), 1.05
(3–6), 0.94 (6–10), 0.86 (10–20), 0.83 (> 20 km): the network is level with the survey on
the short pairs and below it on the long ones. Two things are inside that comparison. The
network routes between fixed representative points, whose paths are about twice the distance
the survey's travellers reported (trunk pairs 7.9 vs 3.9 km): the network has the longer trip
and still the shorter time. And the May 2026 speeds are, per caveat 24, in practice all-day
speeds, so a morning-peak slowdown that the 2017/18 travellers experienced is not in them.
**There is, on this evidence, no 2026 uplift to apply to the survey skim:** the measured
2026 network time plus a terminal allowance sits at or below the survey's door-to-door time
everywhere, by 10 % on the trunk. Under `CAR_SOURCE = 'network'` the car generalized cost
therefore falls slightly and the LRT's capture from car with it; the rerun of 26 → 31 → 32
is reported in §6x / §6ac addenda beside the central case, and the survey time stays the
default until the provider's aggregation is clarified.

**Outputs.**

- `Output/gc/car_ivt_network_area_v2.csv` — the 25 × 25 car time at the 07:00 speeds, kerb to
  kerb including the connectors (minutes; diagonal 0); `car_ivt_network_fastest_hour_area_v2.csv`
  the same at each link's fastest hour; `car_km_network_area_v2.csv` the path length;
  `car_path_fallback_share_area_v2.csv` the share of each path on segments with the assumed
  speed; `car_network_area_connectors.csv` the connector length per area;
  `car_network_segment_coverage.csv` the transfer's coverage of the street network.
- `Output/gc/car_network_skim_sensitivity.csv` — the trunk / all-pair means under the fallback
  variants, the fastest hour and the own-topology cross-check.
- `Output/gc/car_network_vs_survey_pairs.csv`, `car_network_vs_survey_summary.csv` — the pair
  comparison and its summary by distance band, trunk and measured cells; figure
  `Output/figures/car_network_vs_survey.png`.
- Step 26 reads `car_ivt_network_area_v2.csv` under `CAR_SOURCE = 'network'` (car IVT =
  network time + 3 min terminal; intra-area time kept from the survey) and writes to
  `Output/skims/car_network/`; steps 31 and 32 follow with `GC_SOURCE_DIR` / `SK_DIR` set to
  that directory.

**Limits.** The speeds are the delivery's, with caveat 24 (flat hourly profile) in force: the
07:00 skim is in practice a May 2026 all-day skim. The assumed 30 km/h on the segments without
a transferred speed and the 20 km/h connectors are stated, not measured; their weight is in the
sensitivity table. The network time is a fastest path between fixed representative points
with no explicit intersection delay; the survey time is reported, door to door, between the
travellers' actual addresses, and includes everything they counted. The ratio is an uplift
only to the extent that the terminal allowance closes that gap, and on the short pairs it
plainly does not (section 4).